In [0]:
# MUST be first line
spark.sql("USE CATALOG health_care_2026")

db = "healthcare_fraud_demo"

# In Unity Catalog use SCHEMA not DATABASE
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {db}")
spark.sql(f"USE {db}")

base_path = "/Volumes/health_care_2026/healthcare_fraud_demo/healthcare_fraud"

checkpoint = f"{base_path}/checkpoints"
schema_loc = f"{base_path}/schema"

print(checkpoint)
print(schema_loc)

In [0]:
# -----------------------------------------
# LOAD TRUSTED SILVER DATASETS
# -----------------------------------------

claims = spark.table(f"{db}.silver_claims").filter("is_active = true")
providers = spark.table(f"{db}.silver_providers")
patients = spark.table(f"{db}.silver_patients")
payments = spark.table(f"{db}.silver_payments")

print("Silver tables loaded")


In [0]:
# -----------------------------------------
# BUILD ENRICHED CLAIM BUSINESS DATASET
# -----------------------------------------

gold_claim_summary = (
    claims
    .join(providers, "provider_id", "left")
    .join(patients, "patient_id", "left")
    .join(payments, "claim_id", "left")
)

print("Claim summary prepared")


In [0]:
# -----------------------------------------
# FRAUD RULE ENGINE
# -----------------------------------------

fraud_df = (
    gold_claim_summary

    # Rule 1: unpaid claim
    .withColumn("rule_unpaid",
        when(col("paid_amount").isNull(),1).otherwise(0)
    )

    # Rule 2: abnormal variance
    .withColumn("rule_high_variance",
        when(
            col("paid_amount").isNotNull() &
            (col("claim_amount") > col("paid_amount")*5),
            1
        ).otherwise(0)
    )

    # Rule 3: high-risk provider
    .withColumn("rule_high_risk_provider",
        when(col("risk_score") > 0.8,1).otherwise(0)
    )

    # Rule 4: extreme billing amount
    .withColumn("rule_extreme_claim",
        when(col("claim_amount") > 100000,1).otherwise(0)
    )
)

print("Fraud rules applied")


In [0]:
# -----------------------------------------
# GENERATE FINAL FRAUD FLAGS & BUSINESS METRICS
# -----------------------------------------

fraud_df = (
    fraud_df

    # final fraud flag
    .withColumn(
        "fraud_flag",
        when(
            (col("rule_unpaid")==1) |
            (col("rule_high_variance")==1) |
            (col("rule_high_risk_provider")==1) |
            (col("rule_extreme_claim")==1),
            "YES"
        ).otherwise("NO")
    )

    # reason
    .withColumn(
        "fraud_reason",
        when(col("rule_unpaid")==1,"Payment missing")
        .when(col("rule_high_variance")==1,"Claim >> Payment variance")
        .when(col("rule_high_risk_provider")==1,"High risk provider")
        .when(col("rule_extreme_claim")==1,"Extreme claim amount")
        .otherwise("No fraud detected")
    )

    # business-friendly risk %
    .withColumn(
        "risk_score_pct",
        concat(round(col("risk_score")*100,2), lit("%"))
    )
)

print("Final fraud output ready")


In [0]:
# -----------------------------------------
# FINAL CURATED BUSINESS SCHEMA
# -----------------------------------------

gold_claim_summary_final = gold_claim_summary.select(
    "claim_id",
    "provider_id",
    "patient_id",
    "claim_amount",
    "claim_date",
    "paid_amount",
    "name",
    "specialty",
    "risk_score"
)

fraud_df_final = fraud_df.select(
    "claim_id",
    "provider_id",
    "patient_id",
    "claim_amount",
    "claim_date",
    "paid_amount",
    "name",
    "specialty",
    "risk_score",
    "risk_score_pct",
    "fraud_flag",
    "fraud_reason"
)

print("Curated schemas prepared")


In [0]:
# # -----------------------------------------
# # DROP OLD GOLD TABLES (SAFE REBUILD)
# # -----------------------------------------

# spark.sql(f"DROP TABLE IF EXISTS {db}.gold_claim_summary")
# spark.sql(f"DROP TABLE IF EXISTS {db}.gold_fraud_signals")

# print("Old Gold tables removed")


In [0]:
# -----------------------------------------
# WRITE CURATED GOLD DATASETS
# -----------------------------------------

gold_claim_summary_final.write \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(f"{db}.gold_claim_summary")

fraud_df_final.write \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(f"{db}.gold_fraud_signals")


In [0]:
%sql
-- suspicious claims
SELECT claim_id, claim_amount, fraud_flag, fraud_reason,paid_amount
FROM healthcare_fraud_demo.gold_fraud_signals
WHERE fraud_flag = 'No'
ORDER BY claim_amount DESC;



In [0]:
%sql
select * from healthcare_fraud_demo.gold_fraud_signals

In [0]:
%sql
select * from healthcare_fraud_demo.gold_claim_summary

In [0]:
%sql
-- risky providers
SELECT provider_id, name, risk_score_pct, COUNT(*) suspicious_claims
FROM healthcare_fraud_demo.gold_fraud_signals
WHERE fraud_flag = 'YES'
GROUP BY provider_id, name, risk_score_pct
ORDER BY suspicious_claims DESC;
